# SemMap — nSBM → Graph Grammar → Huffman, ConceptNet-10k / 100k

Эксперимент сравнивает **реальные ZIP-архивы**: компактный бинарный baseline, nSBM-guided exact grammar (varint и Huffman), обычный structural-proposal и контроль с перемешанными nSBM-блоками.

**Вычисления:** graph-tool nSBM — CPU/OpenMP (до доступного числа ядер), exact VF2 — параллельные CPU-процессы, разреженное извлечение ego — CPU-потоки, GPU T4/L4 — CuPy для расчёта степеней и блочных W-проекций. GPU **не обучает** nSBM, не выполняет VF2. Значения MDL не являются размером файла.

Режим `PROFILE="10k"` — первая версия; для перехода к `100k` изменить профиль и указать подготовленный 100k TSV из того же пятиколоночного формата. Большой прогон может потребовать High-RAM Colab. Запуски можно возобновлять с Drive, сохраняя тот же Git SHA и конфигурацию.

In [ ]:
# 1. Единственная конфигурационная ячейка
from pathlib import Path
REPO_URL = "https://github.com/SemanticMap/semgraphex.git"
REPO_BRANCH = "feature/nsbm-huffman-grammar-cn10k"
SOURCE_REVISION = "02a03e035e0268e96544f59895752fc22f60f97a"
PROFILE = "10k"  # "10k" | "100k"
RUN_MODE = "NEW"  # "NEW" | "RESUME"; не меняйте при изменении кода/данных без нового RUN_ID
RUN_ID = f"nsbm-huffman-{PROFILE}-v3"
SEED = 42
CPU_BUDGET = 12   # верхняя граница, фактическая учитывает cgroup/affinity
RESTARTS = 3
METHODS = "nsbm,plain,shuffled,wishart,nsbm_wishart" if PROFILE == "10k" else "nsbm,shuffled,wishart"
GPU_REQUIRED = True  # T4/L4, False только для CPU-проверки
DISCOVERY = None  # профиль: 800 для 10k, 3000 для 100k
MAX_EGO_NODES = 7
MIN_FIGURE_NODES = 3
MIN_SUPPORT = 3
MAX_RECURSIVE_LEVELS = 3
SYNC_INTERVAL_SECONDS = 120
DRIVE_PROJECT = Path("/content/drive/MyDrive/SemanticMap")
INPUT_PATHS = {
    "10k": DRIVE_PROJECT / "colab/hypothesis/inputs/h1-cn10k-typed-full-01/conceptnet_en_10k_relations.tsv",
    "100k": DRIVE_PROJECT / "colab/hypothesis/inputs/conceptnet_en_100k_relations.tsv",
}
assert PROFILE in ("10k", "100k") and RUN_MODE in ("NEW", "RESUME")

In [ ]:
# 2. Ресурсы и реальный CPU-бюджет
import os, sys, shutil, subprocess, json, hashlib, time

def allowed_cpus():
    n = os.cpu_count() or 1
    if hasattr(os, "sched_getaffinity"):
        n = min(n, len(os.sched_getaffinity(0)))
    quota_file = Path("/sys/fs/cgroup/cpu.max")
    if quota_file.exists():
        quota, period = quota_file.read_text().split()
        if quota != "max":
            n = min(n, max(1, (int(quota) + int(period) - 1)//int(period)))
    return max(1, n)

CPU_EFFECTIVE = min(CPU_BUDGET, allowed_cpus())
gpu = (subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
       if shutil.which("nvidia-smi") else subprocess.CompletedProcess([], 1, "", ""))
GPU_VISIBLE = gpu.returncode == 0 and "GPU" in gpu.stdout
if GPU_REQUIRED and not GPU_VISIBLE:
    raise RuntimeError("Нет GPU. В Colab выберите Runtime → Change runtime type → T4/L4 GPU.")
DEVICE = "cuda" if GPU_REQUIRED else "auto"
free_gb = shutil.disk_usage("/content").free / 2**30
print({"python": sys.version.split()[0], "cpu_available": allowed_cpus(),
       "cpu_budget": CPU_EFFECTIVE, "free_disk_GiB": round(free_gb,1),
       "gpu": gpu.stdout.strip() if GPU_VISIBLE else None})
if free_gb < 7:
    raise RuntimeError("Для micromamba/graph-tool нужно около 7 GiB свободного диска")

In [ ]:
# 3. Drive: длительное хранение, но не вычисление
from google.colab import drive
drive.mount("/content/drive")
PERSISTENT = DRIVE_PROJECT / "colab" / "hypothesis" / "runs" / RUN_ID
SCRATCH = Path("/content/semmap_nsbm_huffman")
REPO = Path("/content/semmap_nsbm_huffman_repo")
ENV_ROOT = Path("/content/semmap_nsbm_huffman_env")
BIN_ROOT = Path("/content/semmap_nsbm_huffman_bin")
RUN_LOCAL = SCRATCH / "runs" / RUN_ID
SCRATCH.mkdir(parents=True, exist_ok=True)
print("Run:", RUN_ID, "\nDrive:", PERSISTENT)

In [ ]:
# 4. Воспроизводимый checkout ровно одной ревизии
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "75", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "75",
                    "origin", REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_REVISION], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert HEAD == SOURCE_REVISION
print("PINNED:", HEAD)

In [ ]:
# 5. graph-tool в изолированном Python 3.12; GPU CuPy отдельно
BIN_ROOT.mkdir(parents=True, exist_ok=True)
MAMBA = BIN_ROOT / "bin" / "micromamba"
ENV_PY = ENV_ROOT / "bin" / "python"
if not MAMBA.exists():
    subprocess.run(["bash", "-lc", "set -euo pipefail; curl -fsSL "
      "https://micro.mamba.pm/api/micromamba/linux-64/latest | "
      f"tar -xj -C {BIN_ROOT} bin/micromamba"], check=True)
mm_env = dict(os.environ, MAMBA_ROOT_PREFIX="/content/semmap_nsbm_huffman_mamba")
if not ENV_PY.is_file():
    subprocess.run([str(MAMBA), "create", "-y", "-p", str(ENV_ROOT),
        "-c", "conda-forge", "--strict-channel-priority",
        "python=3.12", "graph-tool-base=2.98", "numpy", "scipy",
        "scikit-learn", "networkx", "pyyaml", "psutil", "threadpoolctl",
        "pytest", "pip"], check=True, env=mm_env)
subprocess.run([str(ENV_PY), "-m", "pip", "install", "--no-deps", "-e", str(REPO)], check=True)
if GPU_REQUIRED:
    check_cupy = subprocess.run([str(ENV_PY), "-c",
        "import cupy as cp; print(cp.cuda.runtime.getDeviceCount())"], capture_output=True)
    if check_cupy.returncode:
        subprocess.run([str(ENV_PY), "-m", "pip", "install", "cupy-cuda12x>=13,<15"], check=True)
    subprocess.run([str(ENV_PY), "-c",
        "import cupy as cp; assert cp.cuda.runtime.getDeviceCount()>0; "
        "print('CUDA CuPy ready:', cp.cuda.runtime.getDeviceProperties(0)['name'])"], check=True)
subprocess.run([str(ENV_PY), "-c",
    "import graph_tool.all as gt; print('graph-tool:', gt.__version__)"], check=True)

In [ ]:
# 6. Быстрый smoke (без настоящего обучения nSBM)
check_env = dict(os.environ, OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
                 NUMEXPR_NUM_THREADS="1", PYTHONHASHSEED="0")
subprocess.run([str(ENV_PY), "-m", "pytest", "-q",
    "tests/test_grammar_huffman.py", "tests/test_nsbm_huffman_experiment.py",
    "tests/test_nsbm_checkpoint_independence.py", "tests/test_nsbm_recursive_codec.py",
    "tests/test_nsbm_measured_selection.py", "tests/test_nsbm_fit_checkpoint.py",
    "tests/test_h1_runner.py"],
    cwd=REPO, env=check_env, check=True)

In [ ]:
# 7. Подготовка входа в локальном SSD (никогда не считаем из Drive)
INPUT = Path(INPUT_PATHS[PROFILE])
if not INPUT.is_file():
    raise FileNotFoundError(f"Не найден подготовленный {PROFILE} ConceptNet TSV: {INPUT}. "
        "Укажите путь в INPUT_PATHS. Для 100k нужен репрезентативно отобранный 100k TSV, "
        "не первые 100k строк большого дампа.")
STAGED = SCRATCH / f"input_{PROFILE}.tsv"
if not STAGED.is_file() or STAGED.stat().st_size != INPUT.stat().st_size:
    shutil.copy2(INPUT, STAGED)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()
INPUT_HASH = file_sha256(STAGED)
if file_sha256(INPUT) != INPUT_HASH:
    raise RuntimeError("Drive/local input checksum differs")
print("Input staged:", STAGED, "size MiB:", round(STAGED.stat().st_size/2**20,1),
      "SHA256:", INPUT_HASH)
if RUN_MODE == "RESUME":
    if not PERSISTENT.is_dir():
        raise FileNotFoundError(f"No persisted checkpoint: {PERSISTENT}")
    shutil.copytree(PERSISTENT, RUN_LOCAL, dirs_exist_ok=True)
else:
    if RUN_LOCAL.exists() and any(RUN_LOCAL.iterdir()):
        raise RuntimeError("NEW требует пустую локальную папку; выберите новый RUN_ID")
    if PERSISTENT.exists() and any(PERSISTENT.iterdir()):
        raise RuntimeError("NEW не должен перезаписывать существующий Drive run")

In [ ]:
# 8. Запуск с периодическим сохранением проверенных файлов в Drive
# Не использовать subprocess.run(...capture_output=True) для многчасовых процессов.
# Временные .tmp и COMPLETED во время исполнения не публикуются.

def sync_checkpoint(final=False):
    PERSISTENT.mkdir(parents=True, exist_ok=True)
    for source in sorted(RUN_LOCAL.rglob("*")):
        if not source.is_file() or source.name == "COMPLETED" or source.name.endswith(".tmp"):
            continue
        target = PERSISTENT / source.relative_to(RUN_LOCAL)
        target.parent.mkdir(parents=True, exist_ok=True)
        if (not target.exists() or target.stat().st_size != source.stat().st_size or
            target.stat().st_mtime_ns != source.stat().st_mtime_ns):
            shutil.copy2(source, target)
    if final and (RUN_LOCAL/"COMPLETED").is_file():
        shutil.copy2(RUN_LOCAL/"COMPLETED", PERSISTENT/"COMPLETED")

cmd = [str(ENV_PY), "-m", "semmap_haken.hypothesis.grammar_experiment",
       "--input", str(STAGED), "--output", str(RUN_LOCAL),
       "--profile", PROFILE, "--mode", RUN_MODE, "--seed", str(SEED),
       "--restarts", str(RESTARTS), "--cpu-budget", str(CPU_EFFECTIVE),
       "--device", DEVICE, "--methods", METHODS,
       "--max-ego-nodes", str(MAX_EGO_NODES),
       "--min-figure-nodes", str(MIN_FIGURE_NODES),
       "--min-support", str(MIN_SUPPORT),
       "--max-recursive-levels", str(MAX_RECURSIVE_LEVELS)]
if DISCOVERY is not None:
    cmd += ["--discovery", str(DISCOVERY)]
LOG = SCRATCH / (RUN_ID + ".log")
with LOG.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, cwd=REPO, env=check_env,
                            stdout=log, stderr=subprocess.STDOUT)
    try:
        while True:
            try:
                proc.wait(timeout=SYNC_INTERVAL_SECONDS)
                break
            except subprocess.TimeoutExpired:
                sync_checkpoint(final=False)
                print("Checkpoint synced; process running. Run:", RUN_ID)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        raise
    finally:
        sync_checkpoint(final=proc.poll() == 0)
if proc.returncode != 0:
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-50:]))
    raise RuntimeError(f"Experiment exited with code {proc.returncode}")
assert (PERSISTENT / "COMPLETED").is_file()
print("RESULT:", "\n".join(LOG.read_text(errors="replace").splitlines()[-15:]))

In [ ]:
# 9. Итог: фактический размер архивов (а не nSBM MDL)
report = json.loads((PERSISTENT/"report.json").read_text(encoding="utf-8"))
print("ConceptNet:", report["counts"])
print("CPU:", report["cpu_budget"], "| GPU:", report["degree_backend"],
      "| W:", report["projection"])
print("Baseline bundle:", report["baseline"]["full_bundle_bytes"], "bytes")
for method, row in report["methods"].items():
    print(method, "selected:", row["selection"]["accepted_figures"])
    for coding, ratio in report["ratios_full_bundle"][method].items():
        print("  ", coding, "R =", round(ratio, 4),
              "full ZIP:", row["coding"][coding]["full_bundle_bytes"])
print("Report:", PERSISTENT/"report.json")
print("COMPLETED:", (PERSISTENT/"COMPLETED").exists())
if report.get("recursive"):
    rec = report["recursive"]
    print("Recursive hierarchy:", rec["accepted_levels"], "levels;",
          rec["archive_bytes"], "bytes; status:", rec["stop_reason"])
    print("Recursive archive:", rec["archive"])


### Интерпретация и дальнейший шаг

- `R = full_bundle_bytes / full_binary_baseline_bytes`; **меньше 1 лучше**. Включены одинаковые идентификаторы понятий и утверждений.
- Сравнивайте `nsbm` в первую очередь с `shuffled` (одинаковая двухрежимная схема поиска), и дополнительно с `plain`.
- Научный вывод об эффективности требует нескольких независимых выборок, повторов и сопоставимого Wishart baseline. Эти результаты не симулируются в ноутбуке.
- После успешного 10k переключите в начале `PROFILE="100k"`, настройте `INPUT_PATHS["100k"]` и выберите отдельный `RUN_ID`. Обязательно используйте `NEW` для другого размера графа.